# 2.4 DataFrame Exploration

When working with geoscientific data in machine learning, the quality and structure of your data are key factors in building reliable models. An initial preparation for AI-ready data is to perform initial explortion of the data set. In this lecture, we will walk through a typical data preparation pipeline using a pandas DataFrame, focusing on cleaning and transforming the data to be ready for modeling. The example will highlight reading data, checking correlations, handling missing values (NaNs), and removing zeros where appropriate.

## Read and Explore the data

We will download a Kaggle data set:

https://www.kaggle.com/datasets/lucidlenn/sloan-digital-sky-survey




https://www.kaggle.com/code/alanabd/skyserver-sql2-27-2018-ile-ml-ve-cv/input

In [ ]:
import pandas as pd

!wget "https://raw.githubusercontent.com/UW-MLGEO/MLGeo-dataset/refs/heads/main/data/Skyserver_SQL2_27_2018 6_51_39 PM.csv"

In [ ]:
# df = pd.read_csv(r"path\to\file.csv") # Windows
df = pd.read_csv(r"Skyserver_SQL2_27_2018 6_51_39 PM.csv")

In [ ]:
# Get the first few rows of the dataset
df.head()

In [ ]:
# what datatypes are in the dataset
df.info()

It looks like attribute ``class`` is a string of characters, others are numerical values.

In [ ]:
# Summary statistics
df.describe()


##  Handling Missing Values (NaNs) & Zeros

Geoscience datasets often contain missing values (e.g., due to sensor malfunctions or data collection gaps) and zeros (which may or may not be meaningful depending on the context). You'll need to treat these cases carefully.


Missing data is common in geoscientific applications. In a pandas DataFrame, missing values are typically represented as NaNs. You can handle NaNs in several ways depending on the context:

* **Remove rows/columns with NaNs**: If the missing data is minimal or irrelevant, you can drop it.

* **Impute missing values**: For geoscience data, you might fill NaNs with interpolated values, means, or more sophisticated imputation techniques.

* **Zeros**: Zeros can sometimes be valid measurements, like in precipitation data (no rainfall). However, zeros might also indicate missing or incorrect data in some cases. It’s important to distinguish between meaningful zeros and errors.

In [ ]:
# Check for missing values (NaNs)
print(df.isnull().sum())


In [ ]:

# Check for zeros
print((df == 0).sum())


In [ ]:
# Option 1: Remove rows with NaNs
df_cleaned = df.dropna()

In [ ]:
df.replace([0, float('inf'), -float('inf')], pd.NA, inplace=True)
df_cleaned.replace([0, float('inf'), -float('inf')], pd.NA, inplace=True)

## Final Data Check
After cleaning the data, it’s crucial to perform a final check before feeding it into a machine learning model.

In [ ]:
# Final check for NaNs and zeros
print(df_cleaned.isnull().sum())

In [ ]:
# Find rows where redshift is zero
redshift_zeros = df_cleaned[df_cleaned['redshift'].isnull()]
print(redshift_zeros)

In [ ]:
df_cleaned['redshift'].replace('<NA>', pd.NA, inplace=True)
df_cleaned.dropna(subset=['redshift'], inplace=True)

In [ ]:
df_cleaned.info()

In [ ]:
df_cleaned.to_csv('cleaned_data.csv', index=False)
df_cleaned.describe()

## 2. Correlation Analysis
In machine learning, understanding the relationships between features can provide valuable insights. For example, in geosciences, soil moisture might be correlated with precipitation or vegetation indices. You can use a correlation matrix to check for such relationships.

First, our data frame contains ``object`` that are non numerical values (characters), so to perform the correlation analysis on numerical data, we first have to only look at numerical value. Create a new data frame that only retains numerical values.

In [ ]:
# Create a new data frame that only retains numerical values
df_numerical = df_cleaned.select_dtypes(exclude=['object'])
df_numerical.head()

In [ ]:
# Calculate correlation matrix
corr_matrix = df_numerical.corr()

In [ ]:
!pip install seaborn
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm',cbar=False)
plt.show()


Look for high correlations, which can indicate redundancy among features. In cases where variables are highly correlated, you may decide to drop one to avoid multicollinearity.

The **Spearman rank** correlation coefficient is a non-parametric measure of rank correlation that assesses the strength and direction of the monotonic relationship between two variables. It is particularly useful in geoscientific data analysis for several reasons:

In [ ]:
# Calculate Spearman correlation matrix
spearman_corr_matrix = df_numerical.corr(method='spearman')

# Plot the Spearman correlation matrix
plt.figure(figsize=(15, 12))
sns.heatmap(spearman_corr_matrix, annot=True, cmap='coolwarm', cbar=False)
plt.show()

## Feature distribution

We can now explore how interesting data (features) are distributed for the three classes.

In [ ]:
# Filter the data for each class
galaxy = df_cleaned[df_cleaned['class'] == 'GALAXY']
qso = df_cleaned[df_cleaned['class'] == 'QSO']
star = df_cleaned[df_cleaned['class'] == 'STAR']

# Plot histograms for each band and class
bands = ['u', 'g', 'r', 'i', 'z']
plt.figure(figsize=(15, 10))

for i, band in enumerate(bands):
    plt.subplot(2, 3, i+1)
    sns.histplot(galaxy[band], kde=True, color='blue', label='GALAXY', stat='density')
    sns.histplot(qso[band], kde=True, color='green', label='QSO', stat='density')
    sns.histplot(star[band], kde=True, color='red', label='STAR', stat='density')
    plt.title(f'Distribution of {band} band')
    plt.legend()

plt.tight_layout()
plt.show()

Are there features that could allow us to discriminate between the objects?

Some distrubutions look normal, some look skewed, how to characterize and quantify data distributions?

In [ ]:
from scipy.stats import skew, kurtosis

# Function to calculate summary statistics
def summarize_distribution(df, features):
    summary = {}
    for feature in features:
        summary[feature] = {
            'mean': df[feature].mean(),
            'median': df[feature].median(),
            'std': df[feature].std(),
            'skewness': skew(df[feature]),
            'kurtosis': kurtosis(df[feature])
        }
    return pd.DataFrame(summary)

# Features to summarize
features = ['u', 'g', 'r', 'i', 'z']

# Summarize distributions for each class
summary_galaxy = summarize_distribution(galaxy, features)
summary_qso = summarize_distribution(qso, features)
summary_star = summarize_distribution(star, features)

# Display the summaries
print("Galaxy Summary:")
print(summary_galaxy)
print("\nQSO Summary:")
print(summary_qso)
print("\nStar Summary:")
print(summary_star)

## 4. Exercise 